# 🌐 Notebook 4: Cross-Dataset Generalization Evaluation
This evaluation-only notebook directly answers the CRITIC question: **"Is it better to train on Crack500 and test on DeepCrack? How do we solve cross-dataset overfitting?"**

It evaluates all baseline and KD models trained on **Crack500** directly on the **DeepCrack** test split (and vice versa) without fine-tuning, quantifying domain transfer gap and KD generalization gains.


In [ ]:
!mkdir -p data/datasets scripts


In [ ]:
!pip install -q ultralytics pandas pyyaml


In [ ]:
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

# Define models to evaluate cross-dataset
eval_matrix = [
    # Train: Crack500 -> Test: DeepCrack
    {"name": "v8_crack500_baseline", "train": "Crack500", "test_data": "data/datasets/deepcrack_yolo/dataset.yaml", "ckpt": "runs/baselines/v8_crack500_baseline/weights/best.pt", "type": "Baseline"},
    {"name": "v11_crack500_baseline", "train": "Crack500", "test_data": "data/datasets/deepcrack_yolo/dataset.yaml", "ckpt": "runs/baselines/v11_crack500_baseline/weights/best.pt", "type": "Baseline"},
    {"name": "v11_crack500_full_kd", "train": "Crack500", "test_data": "data/datasets/deepcrack_yolo/dataset.yaml", "ckpt": "runs/crack_distill_full_kd_box_instance_seg_yolo11n-seg/weights/best.pt", "type": "Full KD"},

    # Train: DeepCrack -> Test: Crack500
    {"name": "v8_deepcrack_baseline", "train": "DeepCrack", "test_data": "data/datasets/crack500_yolo/dataset.yaml", "ckpt": "runs/baselines/v8_deepcrack_baseline/weights/best.pt", "type": "Baseline"},
    {"name": "v11_deepcrack_baseline", "train": "DeepCrack", "test_data": "data/datasets/crack500_yolo/dataset.yaml", "ckpt": "runs/baselines/v11_deepcrack_baseline/weights/best.pt", "type": "Baseline"},
    {"name": "v11_deepcrack_full_kd", "train": "DeepCrack", "test_data": "data/datasets/crack500_yolo/dataset.yaml", "ckpt": "runs/crack_distill_full_kd_box_instance_seg_deepcrack/weights/best.pt", "type": "Full KD"},
]

results = []
for item in eval_matrix:
    ckpt_path = Path(item["ckpt"])
    if ckpt_path.exists() and Path(item["test_data"]).exists():
        model = YOLO(str(ckpt_path))
        metrics = model.val(data=item["test_data"], split="test")
        results.append({
            "Experiment": item["name"],
            "Train Domain": item["train"],
            "Test Domain": "DeepCrack" if "deepcrack" in item["test_data"] else "Crack500",
            "Model Type": item["type"],
            "mAP50-seg": metrics.seg.map50,
            "mAP50-95-seg": metrics.seg.map,
            "mAP50-box": metrics.box.map50,
        })
    else:
        print(f"Skipping {item['name']}: checkpoint or data not found.")

df = pd.DataFrame(results)
print("\n" + "="*70)
print("🌐 CROSS-DATASET GENERALIZATION EVALUATION SUMMARY")
print("="*70)
print(df.to_string(index=False))
